In [4]:
import os
import re
import pandas as pd
import tkinter as tk
from tkinter import filedialog

def consolidar_resultados_publicacion(directorios_raiz):
    """
    Recorre una lista de carpetas buscando '_resumen.txt', extrae datos, 
    escala las áreas por 10^11 y los devuelve en un DataFrame en inglés.
    """
    datos_recopilados = []
    
    for directorio in directorios_raiz:
        print(f"\n🔍 Escaneando directorio: {directorio}...\n")
        for root, dirs, files in os.walk(directorio):
            for file in files:
                if file.endswith("_resumen.txt"):
                    ruta_completa = os.path.join(root, file)
                    nombre_planeta = os.path.basename(root).replace("_", " ")
                    
                    with open(ruta_completa, 'r', encoding='utf-8') as f:
                        contenido = f.read()
                    
                    try:
                        # Extraer datos de conteo con Regex (Zonas)
                        z_estables = int(re.search(r'Total estables:\s*(\d+)', contenido, re.IGNORECASE).group(1))
                        z_inestables = int(re.search(r'Total inestables:\s*(\d+)', contenido, re.IGNORECASE).group(1))
                        # z_col_m1 se extrae pero ya no se añade a la tabla final
                        z_col_m1 = int(re.search(r'Total colisi[oó]n m1:\s*(\d+)', contenido, re.IGNORECASE).group(1))
                        z_col_m2 = int(re.search(r'Total colisi[oó]n m2:\s*(\d+)', contenido, re.IGNORECASE).group(1))
                        
                        # Extraer datos de áreas
                        a_est = float(re.search(r'[AÁaá]rea estable:\s*([0-9\.eE\+\-]+)', contenido).group(1))
                        a_inest = float(re.search(r'[AÁaá]rea inestable:\s*([0-9\.eE\+\-]+)', contenido).group(1))
                        a_col_m2 = float(re.search(r'[AÁaá]rea de colisi[oó]n con m2:\s*([0-9\.eE\+\-]+)', contenido).group(1))
                        
                        # ESCALAR POR 10^11
                        FACTOR_ESCALA = 1e11
                        
                        # Agregar diccionario a la lista (En inglés, con unidades incluidas y sin Col. m1)
                        datos_recopilados.append({
                            "Planet": nombre_planeta,
                            "Stable Zones": z_estables,
                            "Unstable Zones": z_inestables,
                            "Col. m2 Zones": z_col_m2,
                            "Stable Area (10^11 km^2)": a_est / FACTOR_ESCALA,
                            "Unstable Area (10^11 km^2)": a_inest / FACTOR_ESCALA,
                            "Col. m2 Area (10^11 km^2)": a_col_m2 / FACTOR_ESCALA
                        })
                        print(f"✅ Datos extraídos de: {nombre_planeta}")
                    except Exception as e:
                        print(f"❌ Error procesando {file}: {e}")
                    
    return pd.DataFrame(datos_recopilados)

def generar_archivos_publicacion(df, directorio_salida, nombre_salida="Global_Results_Table"):
    """
    Exporta la tabla a formatos útiles (CSV, TXT, LaTeX) en el directorio especificado.
    """
    if df.empty:
        print("La tabla está vacía. No se exportó nada.")
        return

    # Construir rutas completas uniendo el directorio destino y el nombre del archivo
    ruta_csv = os.path.join(directorio_salida, f"{nombre_salida}.csv")
    ruta_txt = os.path.join(directorio_salida, f"{nombre_salida}.txt")
    ruta_tex = os.path.join(directorio_salida, f"{nombre_salida}.tex")

    # 1. Exportar a CSV
    df.to_csv(ruta_csv, index=False)
    
    # 2. Exportar a TXT (Formateado a 2 decimales)
    with open(ruta_txt, "w", encoding='utf-8') as f:
        f.write(df.to_string(index=False, float_format=lambda x: f"{x:.2f}"))
        
    # 3. Exportar a LaTeX 
    df_latex = df.copy()
    
    # Renombrar columnas para agregar la notación matemática en la cabecera de LaTeX
    df_latex.rename(columns={
        "Stable Area (10^11 km^2)": "Stable Area ($10^{11}$ km$^2$)",
        "Unstable Area (10^11 km^2)": "Unstable Area ($10^{11}$ km$^2$)",
        "Col. m2 Area (10^11 km^2)": "Col. m2 Area ($10^{11}$ km$^2$)"
    }, inplace=True)
    
    try:
        # Método moderno de Pandas (>= 1.3)
        # column_format actualizado a 7 columnas (l + 6 r)
        latex_code = df_latex.style.format(precision=2).hide(axis="index").to_latex(
            hrules=True, 
            caption="Orbital stability analysis and corresponding areas.", 
            label="tab:stability_areas",
            column_format="lrrrrrr"
        )
    except AttributeError:
        # Método antiguo de Pandas (< 1.3)
        latex_code = df_latex.to_latex(
            index=False,
            float_format="%.2f",
            column_format="lrrrrrr",
            caption="Orbital stability analysis and corresponding areas.",
            label="tab:stability_areas",
            escape=False # Permite que se imprima el $10^{11}$ sin desconfigurarse
        )
        
    with open(ruta_tex, "w", encoding='utf-8') as f:
        f.write(latex_code)
        
    print(f"\n🎉 Archivos creados con éxito en: {directorio_salida}")
    print(f"Se generaron: {nombre_salida}.csv, .txt y .tex")

if __name__ == "__main__":
    # Ocultar la ventana principal de Tkinter
    root = tk.Tk()
    root.withdraw()
    root.attributes('-topmost', True)

    carpetas_seleccionadas = []
    
    # Ruta base sugerida
    ruta_inicial = r'C:\Users\darkm\Desktop\archivos_Investigación\Investigacion\Troyanos\Resultados_Estrellas_K'
    
    print("Preparando la selección de las 4 carpetas de origen...")
    
    # Pedir exactamente 4 carpetas
    for i in range(1, 5):
        carpeta = filedialog.askdirectory(
            initialdir=ruta_inicial, 
            title=f"Selecciona la carpeta de origen {i} de 4"
        )
        if carpeta:
            carpetas_seleccionadas.append(carpeta)
            print(f"[{i}/4] Seleccionada: {carpeta}")
        else:
            print(f"[{i}/4] Selección omitida o cancelada.")

    # Si se seleccionó al menos una carpeta, procedemos
    if carpetas_seleccionadas:
        print("\nProcesando los directorios seleccionados...")
        tabla_final = consolidar_resultados_publicacion(carpetas_seleccionadas)
        
        if not tabla_final.empty:
            print("\n=== VISTA PREVIA DE LA TABLA ===")
            print(tabla_final.to_string(index=False, float_format=lambda x: f"{x:.2f}"))
            
            # Pedir carpeta de destino
            print("\nSelecciona la carpeta donde deseas GUARDAR los archivos de salida...")
            carpeta_destino = filedialog.askdirectory(
                initialdir=ruta_inicial, 
                title="Selecciona la carpeta de DESTINO para guardar los archivos"
            )
            
            # Si el usuario cierra la ventana de guardar, usar el directorio actual
            if not carpeta_destino:
                print("\nNo se seleccionó destino. Guardando en la carpeta actual del script...")
                carpeta_destino = os.getcwd()
                
            generar_archivos_publicacion(tabla_final, carpeta_destino)
        else:
            print("\nNo se encontraron archivos '_resumen.txt' válidos en las carpetas seleccionadas.")
    else:
        print("\nOperación cancelada. No seleccionaste ninguna carpeta de origen.")

Preparando la selección de las 4 carpetas de origen...
[1/4] Seleccionada: C:/Users/darkm/Desktop/archivos_Investigación/Investigacion/Troyanos/Resultados_Estrellas_F
[2/4] Seleccionada: C:/Users/darkm/Desktop/archivos_Investigación/Investigacion/Troyanos/Resultados_Estrellas_G
[3/4] Seleccionada: C:/Users/darkm/Desktop/archivos_Investigación/Investigacion/Troyanos/Resultados_Estrellas_K
[4/4] Seleccionada: C:/Users/darkm/Desktop/archivos_Investigación/Investigacion/Troyanos/Resultados_Estrellas_M

Procesando los directorios seleccionados...

🔍 Escaneando directorio: C:/Users/darkm/Desktop/archivos_Investigación/Investigacion/Troyanos/Resultados_Estrellas_F...

✅ Datos extraídos de: DMPP-1 b
✅ Datos extraídos de: DMPP-1 c
✅ Datos extraídos de: DMPP-1 d
✅ Datos extraídos de: DMPP-1 e
✅ Datos extraídos de: DMPP-2 b
✅ Datos extraídos de: DMPP-2 c
✅ Datos extraídos de: DMPP-2 d
✅ Datos extraídos de: HD 22946 d
✅ Datos extraídos de: TOI-411 b
✅ Datos extraídos de: TOI-411 c
✅ Datos extraído